# Pandas — Phase 8: Time Series, Text, Outliers & Exporting
### Credit Card Risk Analysis Track

**Topics in this phase:**
33. Time Series Tools — `.resample()`, `.rolling()`, `.expanding()`, `.shift()`
34. Advanced String Manipulation — `.str.split()`, `.str.extract()`, `.str.strip()`, `.str.len()`
35. Outlier Detection — IQR method, z-score method
36. Exporting Data — `.to_csv()`, `.to_excel()`, `.to_json()`
37. MultiIndex Basics — slicing, `.xs()`

**New dataset for this phase:** `daily_portfolio_metrics.csv` — 730 days (2 years) of daily portfolio activity: `total_applications`, `total_loan_volume`, `default_count`, with realistic weekday/weekend patterns, a gentle upward trend, and seasonal swings. Needed because `loan_applications.csv` has one row per application, not one row per day — meaningful time series work needs an actual daily series. Place it alongside `loan_applications.csv` in the same folder as this notebook.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual datasets before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

## Topic 33: Time Series Tools

These tools only make sense once you have a genuine time series — one row per time period, indexed by an actual datetime. Run this setup cell first:

In [ ]:
daily = pd.read_csv("daily_portfolio_metrics.csv", parse_dates=["date"])
daily = daily.set_index("date")
print(daily.shape)
daily.head()

**Q1.** Confirm `daily`'s index is a real datetime index by printing its `.index.dtype` alongside the DataFrame's shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(daily.shape, daily.index.dtype)

**Q2.** Use `.resample("ME")` (month-end) to collapse the daily `total_applications` into **monthly totals**, into `monthly_applications`, chaining `.sum()`. This is a groupby in disguise — it buckets by calendar month, exactly like `.groupby()` would bucket by category.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
monthly_applications = daily["total_applications"].resample("ME").sum()
print(monthly_applications.head())

**Q3.** Resample `total_loan_volume` to **weekly averages** instead, using `.resample("W").mean()`, into `weekly_avg_volume`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
weekly_avg_volume = daily["total_loan_volume"].resample("W").mean()
print(weekly_avg_volume.head())

**Q4.** `.rolling()` computes a **moving window** statistic — useful for smoothing out day-to-day noise. Compute `rolling_7day_apps`: the 7-day rolling mean of `total_applications`, using `.rolling(7).mean()`. Print the first 10 values — notice the first 6 are `NaN`, since there aren't 7 days of history yet to average over.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
rolling_7day_apps = daily["total_applications"].rolling(7).mean()
print(rolling_7day_apps.head(10).tolist())

**Q5.** Compute `rolling_30day_defaults`: a rolling 30-day **sum** (not mean) of `default_count` — "how many defaults happened in the trailing month," a genuinely common risk monitoring metric. Print the last 5 values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
rolling_30day_defaults = daily["default_count"].rolling(30).sum()
print(rolling_30day_defaults.tail(5).tolist())

**Q6.** `.expanding()` is like `.rolling()` but the window keeps **growing** — every point is the statistic over *all* history up to that point, not just a fixed recent window. Compute `expanding_avg_volume`: the running (cumulative) average of `total_loan_volume` from day one onward, using `.expanding().mean()`. Print the first 5 values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
expanding_avg_volume = daily["total_loan_volume"].expanding().mean()
print(expanding_avg_volume.head(5).tolist())

**Q7.** `.shift()` moves values forward or backward in time — the standard way to build a "lag" feature. Add `applications_prev_day`: each day's value is **yesterday's** `total_applications`, using `.shift(1)`. Then add `applications_day_over_day_change` using `.diff()` (which is really just `original - original.shift(1)`, done for you in one call). Print the first 5 rows of all three columns together.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
daily["applications_prev_day"] = daily["total_applications"].shift(1)
daily["applications_day_over_day_change"] = daily["total_applications"].diff()
print(daily[["total_applications", "applications_prev_day", "applications_day_over_day_change"]].head(5))

## Topic 34: Advanced String Manipulation

Back to `loan_applications.csv`. Run this setup cell first — it adds a synthetic free-text `applicant_notes` field (the kind of messy field a legacy system export often has) and a padded version of `home_ownership` with stray whitespace, so there's something real to practice on:

In [ ]:
df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])

rng = np.random.default_rng(21)
ref_ids = [f"JD{n:05d}" for n in rng.integers(10000, 99999, size=len(df))]
phones = [f"555-{n:04d}" for n in rng.integers(1000, 9999, size=len(df))]
regions = rng.choice(["North", "South", "East", "West"], size=len(df))
df["applicant_notes"] = [
    f"Ref:{r} Phone:{p} Region:{g}" for r, p, g in zip(ref_ids, phones, regions)
]

padded_ownership = df["home_ownership"].copy()
pad_idx = rng.choice(len(df), size=20, replace=False)
padded_ownership.iloc[pad_idx] = "  " + padded_ownership.iloc[pad_idx] + "  "
df["home_ownership_padded"] = padded_ownership

df[["applicant_notes", "home_ownership_padded"]].head()

**Q8.** Print the first 3 values of `applicant_notes` to see its format: `"Ref:JD12345 Phone:555-6789 Region:North"`-style strings. The next few questions pull structured fields back out of this.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(df["applicant_notes"].head(3).tolist())

**Q9.** `employment_status` sometimes contains a hyphen (`"Self-Employed"`). Use `.str.split("-", expand=True)` to split every value on the hyphen into **separate columns**, into `employment_split`. Print the first 5 rows — values with no hyphen (like `"Employed"`) get `NaN` in the second column.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
employment_split = df["employment_status"].str.split("-", expand=True)
print(employment_split.head(5))

**Q10.** Use `.str.extract()` with a regex capture group to pull the reference ID out of `applicant_notes` into a new `ref_id` column — the pattern `r"Ref:(\w+)"` matches `"Ref:"` followed by word characters, and the parentheses mark what gets extracted.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["ref_id"] = df["applicant_notes"].str.extract(r"Ref:(\w+)")
print(df["ref_id"].head(5).tolist())

**Q11.** Clean `home_ownership_padded` by stripping leading/trailing whitespace with `.str.strip()`, into `home_ownership_clean`. Then confirm two things: how many rows actually had stray whitespace to begin with (compare `home_ownership_padded` to its own `.str.strip()` result), and that `home_ownership_clean` now exactly matches the original clean `home_ownership` column.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["home_ownership_clean"] = df["home_ownership_padded"].str.strip()
has_whitespace_before = (df["home_ownership_padded"] != df["home_ownership_padded"].str.strip()).sum()
print(has_whitespace_before, (df["home_ownership_clean"] == df["home_ownership"]).all())

**Q12.** Add `notes_length`: the character length of every `applicant_notes` value, using `.str.len()`. Print the first 3 values — useful for spotting truncated or unusually short/long free-text entries at a glance.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["notes_length"] = df["applicant_notes"].str.len()
print(df["notes_length"].head(3).tolist())

**Q13.** Extract the phone number out of `applicant_notes` into a `phone_number` column, using `.str.extract()` with the pattern `r"Phone:(\d{3}-\d{4})"` (3 digits, a dash, 4 digits).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["phone_number"] = df["applicant_notes"].str.extract(r"Phone:(\d{3}-\d{4})")
print(df["phone_number"].head(5).tolist())

## Topic 35: Outlier Detection

You've filtered for extreme values before, but always with a threshold you chose by hand. These are the two standard, defensible ways to define "extreme" statistically.

**Q14.** Compute the IQR bounds for `loan_amount`: `q1` (25th percentile), `q3` (75th percentile), `iqr` (`q3 - q1`), and the classic Tukey fence `lower_bound` = `q1 - 1.5 * iqr`, `upper_bound` = `q3 + 1.5 * iqr`. Print all five values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
q1 = df["loan_amount"].quantile(0.25)
q3 = df["loan_amount"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(q1, q3, iqr, lower_bound, upper_bound)

**Q15.** Using the bounds from Q14, build `is_outlier_iqr`: `True` where `loan_amount` falls outside `[lower_bound, upper_bound]`. Print how many rows are flagged.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
is_outlier_iqr = (df["loan_amount"] < lower_bound) | (df["loan_amount"] > upper_bound)
print(is_outlier_iqr.sum())

**Q16.** The other standard approach: compute a z-score for every `loan_amount` value (`(x - mean) / std`), into `z_scores`, then flag `is_outlier_zscore`: `True` where the **absolute** z-score exceeds 3 (more than 3 standard deviations from the mean, a common rule of thumb). Print how many rows are flagged.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loan_mean = df["loan_amount"].mean()
loan_std = df["loan_amount"].std()
z_scores = (df["loan_amount"] - loan_mean) / loan_std
is_outlier_zscore = z_scores.abs() > 3
print(is_outlier_zscore.sum())

**Q17.** Build `comparison`, a small `pd.Series` with two entries — `"iqr_method"` and `"zscore_method"` — holding how many outliers each approach flagged (Q15 and Q16). They usually won't agree exactly: IQR is based on the data's actual quartiles and is more robust to extreme values; the z-score method assumes something closer to a normal distribution and can be thrown off by the very outliers it's trying to detect.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
comparison = pd.Series({"iqr_method": is_outlier_iqr.sum(), "zscore_method": is_outlier_zscore.sum()})
print(comparison)

**Q18.** Rather than dropping outliers outright (losing data), a common compromise is **winsorizing**: capping extreme values at the boundary instead of removing the row. Create `loan_amount_winsorized` using `.clip(lower=lower_bound, upper=upper_bound)` on `loan_amount`. Print how many values actually got changed by the clip.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["loan_amount_winsorized"] = df["loan_amount"].clip(lower=lower_bound, upper=upper_bound)
num_changed = (df["loan_amount_winsorized"] != df["loan_amount"]).sum()
print(num_changed)

## Topic 36: Exporting Data

Every notebook so far has only ever *read* files. A real analysis eventually needs to hand results off — to a colleague, a dashboard, or another script.

**Q19.** Export `df` to `"loan_applications_processed.csv"` using the simplest possible call, `.to_csv(path)`. (No need to print anything meaningful back — just confirm it ran.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df.to_csv("loan_applications_processed.csv")
print("exported")

**Q20.** By default, `.to_csv()` writes the DataFrame's index as an extra column — usually not what you want for a clean handoff file. Export again to `"loan_applications_clean_export.csv"`, this time with `index=False`. Then read it back into `reloaded` and confirm its shape matches `df`'s shape exactly (with `index=True`, the reloaded file would have picked up an extra unnamed column instead).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df.to_csv("loan_applications_clean_export.csv", index=False)
reloaded = pd.read_csv("loan_applications_clean_export.csv")
print(reloaded.shape == df.shape)

**Q21.** Export just the first 50 rows to Excel, `"loan_applications_sample.xlsx"`, with `sheet_name="Sample"` and `index=False`, using `.to_excel()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df.head(50).to_excel("loan_applications_sample.xlsx", sheet_name="Sample", index=False)
print("exported")

**Q22.** Export the first 10 rows to JSON, `"loan_applications_sample.json"`, using `.to_json(path, orient="records", date_format="iso")` — `orient="records"` gives you a list of `{column: value}` objects (the most common, most portable JSON shape for tabular data), and `date_format="iso"` writes any datetime columns as readable ISO date strings instead of pandas' default raw epoch-millisecond numbers.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df.head(10).to_json("loan_applications_sample.json", orient="records", date_format="iso")
print("exported")

**Q23.** You rarely export the raw data itself — usually it's a **summary**. Build `summary_report`: mean and count of `loan_amount` grouped by `loan_status` (using `.agg(["mean", "count"])`), then export it to `"loan_summary_report.csv"`. This is a realistic end-of-pipeline deliverable — a small, readable report someone could open directly in Excel.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
summary_report = df.groupby("loan_status")["loan_amount"].agg(["mean", "count"])
summary_report.to_csv("loan_summary_report.csv")
print(summary_report)

## Topic 37: MultiIndex Basics

You've already produced MultiIndexes without necessarily naming them — a two-key `.groupby()`, `pd.concat(keys=...)` from Phase 6, and pivot tables all create one. Here's how to work with one directly.

**Q24.** Build `multi_summary`: group `df` by **both** `employment_status` and `home_ownership`, with named aggregation for `avg_loan` (mean `loan_amount`) and `count` (count of `application_id`). Print the type of its index and `.index.names` — grouping by two columns automatically produces a `MultiIndex`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
multi_summary = df.groupby(["employment_status", "home_ownership"]).agg(
    avg_loan=("loan_amount", "mean"), count=("application_id", "count")
)
print(type(multi_summary.index), multi_summary.index.names)

**Q25.** `.loc` works on a `MultiIndex` too. Get every row for `"Employed"` (all home-ownership types) using `.loc["Employed"]`, into `employed_only`. Then get one exact combination — `("Employed", "Rent")` — using `.loc[("Employed", "Rent")]`, into `specific_combo`, by passing a tuple matching both index levels.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
employed_only = multi_summary.loc["Employed"]
specific_combo = multi_summary.loc[("Employed", "Rent")]
print(employed_only)
print(specific_combo)

**Q26.** `.loc["Employed"]` (Q25) slices on the **outer** (first) index level easily. To slice on an **inner** level instead — every `"Rent"` row regardless of employment status — use `.xs("Rent", level="home_ownership")`, into `rent_across_all_employment`. This is exactly the situation `.xs()` exists for.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
rent_across_all_employment = multi_summary.xs("Rent", level="home_ownership")
print(rent_across_all_employment)

**Q27.** Reorder the index levels themselves with `.swaplevel()` (making `home_ownership` the outer level and `employment_status` the inner one), then `.sort_index()` so it displays in a sensible order, into `swapped`. Print the first few rows.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
swapped = multi_summary.swaplevel().sort_index()
print(swapped.head())

**Q28.** As a recap tying back to Phase 5: flatten `multi_summary` back into a normal DataFrame with a plain `RangeIndex` using `.reset_index()`, into `flattened`. Print its columns and confirm its index type.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
flattened = multi_summary.reset_index()
print(flattened.columns.tolist(), type(flattened.index))

## ✅ Checkpoint

**What you covered:**
- Time series: `.resample()` for bucketing by calendar period, `.rolling()` for fixed-window statistics, `.expanding()` for cumulative statistics, and `.shift()`/`.diff()` for lag and change features
- Advanced strings: `.str.split(expand=True)`, `.str.extract()` with regex capture groups, `.str.strip()`, `.str.len()`
- Outlier detection: the IQR/Tukey-fence method, the z-score method, comparing the two, and winsorizing with `.clip()` as an alternative to dropping rows
- Exporting: `.to_csv()` (with and without the index), `.to_excel()`, `.to_json()` (with `orient="records"` and `date_format="iso"`), and exporting a summary report rather than raw data
- MultiIndex: how a two-key groupby produces one, `.loc` on the outer level and by tuple, `.xs()` for an inner level, `.swaplevel()`, and `.reset_index()` to flatten back down

**Why it matters for the project:** trend and rolling-window metrics (Topic 33) are how a risk team actually monitors a portfolio month to month, not just as a one-time snapshot; the string and outlier tools round out the data-cleaning toolkit for the messy free-text and extreme-value cases real exports contain; and exporting is the step that turns your analysis into something someone else can actually use.

**With Phases 0-8 complete, that's the full pandas curriculum** — architecture, ingestion, selection, cleaning, transformation, aggregation, merging/dates, reshaping/binning/groupby/query, and time series/strings/outliers/export/MultiIndex. From here, matplotlib and seaborn are the natural next step for actually visualizing everything you've built.